PubMedQA-labeled: до и после починки...


In [1]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/external_benchmarks/pubmedqa'
os.chdir(_NOTEBOOK_DIR)


In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
EXT = Path('..').resolve()
LABELS = ['yes', 'no', 'maybe']
HEDGE = 'maybe'

def read(path):
    d = pd.read_csv(path, sep=';', encoding='utf-8-sig', low_memory=False)
    d.columns = [c.lstrip('\ufeff') for c in d.columns]
    return d[d['case_text'].notna()] if 'case_text' in d.columns else d

def norm(series):
    return series.astype(str).str.strip().str.lower().where(lambda s: s.isin(LABELS))
SOURCES = {'baseline «до»': (EXT / 'medmcqa/baseline_gpt/baseline_pubmedqa_labeled_gpt.csv', 'baseline_decision'), 'baseline «после»': (Path('baseline_gpt/baseline_pubmedqa_labeled_gpt_fixed.csv'), 'baseline_decision'), 'Delphi «до»': (EXT / 'initial_runs/heterogeneous_gpt120_organizer/delphi_pubmedqa_labeled_noroles.csv', 'winner_decision'), 'Delphi без остановки': (Path('heterogeneous_gpt120_organizer/delphi_pubmedqa_labeled_pbase_fixed.csv'), 'winner_decision'), 'Delphi остановка, калибр R1': (Path('heterogeneous_gpt120_organizer/delphi_pubmedqa_labeled_pbase_earlystop.csv'), 'winner_decision'), 'Delphi остановка, калибр R1-R3': (Path('heterogeneous_gpt120_organizer/delphi_pubmedqa_labeled_pbase_earlystop_calib3.csv'), 'winner_decision')}
frames, truth = ({}, None)
for name, (path, col) in SOURCES.items():
    if not Path(path).exists():
        print(f'ПРОПУЩЕН (файла нет): {name} -> {path}')
        continue
    d = read(path)
    key = 'case_id' if 'case_id' in d.columns else 'question'
    frames[name] = pd.Series(norm(d[col]).values, index=d[key].values)
    if truth is None:
        truth = pd.Series(norm(d['final_decision']).values, index=d[key].values)
    print(f'{name:<18} строк {len(d)}')
common = set.intersection(*[set(s.index) for s in frames.values()])
common = sorted(common)
print(f'\nобщих кейсов: {len(common)}')
S = pd.DataFrame({k: v.reindex(common) for k, v in frames.items()})
S['TRUTH'] = truth.reindex(common)
S = S[S['TRUTH'].notna()]
print('распределение истины:', S['TRUTH'].value_counts(normalize=True).round(3).to_dict())


ПРОПУЩЕН (файла нет): baseline «до» -> .\experiments\external_benchmarks\pubmedqa\..\medmcqa\baseline_gpt\baseline_pubmedqa_labeled_gpt.csv
baseline «после»   строк 100
ПРОПУЩЕН (файла нет): Delphi «до» -> .\experiments\external_benchmarks\pubmedqa\..\initial_runs\heterogeneous_gpt120_organizer\delphi_pubmedqa_labeled_noroles.csv


Delphi без остановки строк 100
Delphi остановка, калибр R1 строк 100


Delphi остановка, калибр R1-R3 строк 100

общих кейсов: 100
распределение истины: {'yes': 0.58, 'no': 0.26, 'maybe': 0.16}


In [3]:
rows = []
for name in frames:
    s = S[[name, 'TRUTH']].dropna()
    rows.append({'система': name, 'n': len(s), 'accuracy': round((s[name] == s['TRUTH']).mean(), 4), 'доля maybe': round((s[name] == HEDGE).mean(), 3), 'нераспознано': round(S[name].isna().mean(), 3)})
rows.append({'система': 'ИСТИНА', 'n': len(S), 'accuracy': 1.0, 'доля maybe': round((S['TRUTH'] == HEDGE).mean(), 3), 'нераспознано': 0.0})
head = pd.DataFrame(rows)
head


,система,n,accuracy,доля maybe,нераспознано
0,baseline «после»,100,0.73,0.03,0.0
1,Delphi без остановки,100,0.66,0.08,0.0
2,"Delphi остановка, калибр R1",100,0.64,0.24,0.0
3,"Delphi остановка, калибр R1-R3",100,0.70,0.02,0.0
4,ИСТИНА,100,1.00,0.16,0.0


In [4]:
def mc(a_name, b_name):
    s = S[[a_name, b_name, 'TRUTH']].dropna()
    a = s[a_name] == s['TRUTH']
    b = s[b_name] == s['TRUTH']
    tb = [[int((a & b).sum()), int((a & ~b).sum())], [int((~a & b).sum()), int((~a & ~b).sum())]]
    return {'сравнение': f'{a_name} vs {b_name}', 'n': len(s), 'acc A': round(a.mean(), 4), 'acc B': round(b.mean(), 4), 'b': tb[0][1], 'c': tb[1][0], 'p': mcnemar(tb, exact=True).pvalue}
pairs = [('Delphi остановка, калибр R1-R3', 'baseline «после»'), ('Delphi остановка, калибр R1-R3', 'Delphi остановка, калибр R1'), ('Delphi остановка, калибр R1-R3', 'Delphi без остановки'), ('Delphi без остановки', 'Delphi «до»'), ('baseline «после»', 'baseline «до»'), ('Delphi «до»', 'baseline «до»')]
pd.DataFrame([mc(a, b) for a, b in pairs if a in frames and b in frames])


,сравнение,n,acc A,acc B,b,c,p
0,"Delphi остановка, калибр R1-R3 vs baseline «по...",100,0.7,0.73,4,7,0.548828
1,"Delphi остановка, калибр R1-R3 vs Delphi остан...",100,0.7,0.64,12,6,0.237885
2,"Delphi остановка, калибр R1-R3 vs Delphi без о...",100,0.7,0.66,8,4,0.387695


In [5]:
for name in frames:
    s = S[[name, 'TRUTH']].dropna()
    ct = pd.crosstab(s['TRUTH'], s[name]).reindex(index=LABELS, columns=LABELS).fillna(0).astype(int)
    print(f'\n{name}')
    print(ct.to_string())



baseline «после»
baseline «после»  yes  no  maybe
TRUTH                           
yes                54   2      2
no                  8  18      0
maybe               9   6      1

Delphi без остановки
Delphi без остановки  yes  no  maybe
TRUTH                               
yes                    45   6      7
no                      4  21      1
maybe                   7   9      0



Delphi остановка, калибр R1
Delphi остановка, калибр R1  yes  no  maybe
TRUTH                                      
yes                           42   2     14
no                             4  17      5
maybe                          5   6      5

Delphi остановка, калибр R1-R3
Delphi остановка, калибр R1-R3  yes  no  maybe
TRUTH                                         
yes                              49   8      1
no                                6  20      0
maybe                             7   8      1


In [6]:
new_path = Path('heterogeneous_gpt120_organizer/delphi_pubmedqa_labeled_pbase_fixed.csv')
if new_path.exists():
    d = read(new_path)
    ids = [c.split('__', 1)[1] for c in d.columns if c.startswith('r1_decision__')]
    t = norm(d['final_decision'])
    out = []
    for r in (1, 2, 3):
        for pid in ids:
            col = f'r{r}_decision__{pid}'
            if col not in d.columns:
                continue
            v = norm(d[col])
            m = v.notna()
            out.append({'раунд': r, 'эксперт': pid, 'валидных': int(m.sum()), 'accuracy': round((v[m] == t[m]).mean(), 3), 'доля maybe': round((v[m] == HEDGE).mean(), 3)})
    rounds = pd.DataFrame(out)
    print(rounds.to_string(index=False))
    print('\nсводно по раундам:')
    print(rounds.groupby('раунд')[['accuracy', 'доля maybe']].mean().round(3).to_string())
    print('\nраспределение числа раундов:', d['delphi_rounds_executed'].value_counts().sort_index().to_dict())
    print('причины ранней остановки:', d['delphi_early_stop_reason'].value_counts(dropna=False).to_dict())
else:
    print('прогон панели ещё не сделан')


 раунд            эксперт  валидных  accuracy  доля maybe
     1 deepseekv4_default       100     0.730       0.090
     1    gpt120b_default       100     0.730       0.060
     1       qwen_default       100     0.680       0.100
     2 deepseekv4_default        99     0.667       0.061
     2    gpt120b_default       100     0.680       0.090
     2       qwen_default       100     0.620       0.220
     3 deepseekv4_default        98     0.684       0.071
     3    gpt120b_default       100     0.650       0.090
     3       qwen_default       100     0.630       0.080

сводно по раундам:
       accuracy  доля maybe
раунд                      
1         0.713       0.083
2         0.656       0.124
3         0.655       0.080

распределение числа раундов: {3.0: 100}
причины ранней остановки: {nan: 100}


In [7]:
if new_path.exists():
    r1 = [norm(d[f'r1_decision__{pid}']) for pid in ids]
    allm = np.logical_and.reduce([(x == HEDGE).values for x in r1])
    print(f'единогласное «maybe» на R1: {allm.sum()} из {len(d)} ({allm.mean():.1%})')
    print('  (в старом прогоне: 415 из 1000, то есть 41.5%)')
    if allm.sum():
        sub = t[allm]
        print('  истина в этих кейсах:', sub.value_counts(normalize=True).round(3).to_dict())
        print(f'  доля заведомо проигранных: {(sub != HEDGE).mean():.1%}  (было 84.1%)')


единогласное «maybe» на R1: 0 из 100 (0.0%)
  (в старом прогоне: 415 из 1000, то есть 41.5%)


In [8]:
head.to_csv('pubmedqa_accuracy.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([mc(a, b) for a, b in pairs if a in frames and b in frames]).to_csv('pubmedqa_paired_tests.csv', index=False, encoding='utf-8-sig')
